# AI Newsroom Studio — Pipeline Notebook
10-agent pipeline: HackerNews trends → fact-check → script → video → YouTube Shorts.

**Agents built:** Agent 1 (Trend Hunter) · Agent 2 (Context Researcher) · Agent 3 (Fact Checker)

---
## Setup: Shared State

In [ ]:
class NewsroomState(TypedDict):
    stories: dict   # per-story (Agents 1-4)
    script:  dict   # NEW top-level key (Agent 5 adds this)
    # script = {
    #   "full_text":    str,   # complete script
    #   "word_count":   int,   # verified count
    #   "est_duration": str,   # "74s"
    #   "sections":     dict,  # parsed labels
    #   "stories_used": list,  # [1, 2, 3] selection ranks
    #   "attempt":      int,   # 1 or 2 (audit trail)
    # }

---
## Agent 1: Trend Fetcher
Fetches top HackerNews stories and scores them by velocity.

**Velocity** = (upvotes + comments×2) / age_hrs — rewards recent engagement.

In [2]:
from agents.agent1 import fetch_trends
import re

def make_story_id(title: str) -> str:
    """Slugify title to a stable dict key. e.g. 'Melody India Italy' → 'melody-india-italy'"""
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

# AGENT 1 NODE
def trend_hunter_node(state: NewsroomState) -> dict:
    trends = fetch_trends()
    stories = {make_story_id(t["title"]): t for t in trends}
    return {"stories": stories}

In [3]:
# Run Agent 1
call = trend_hunter_node(NewsroomState(stories={}))
print(f"Stories fetched: {len(call['stories'])}")
for sid, s in call['stories'].items():
    print(f"  {s['velocity']:>6.1f} vel | {sid}")

Stories fetched: 8
   393.2 vel | microsoft-fire-idtech-team-at-id-software
   210.6 vel | chat-control-passed-first-round-in-eu-parliament
   184.6 vel | 98-isn-t-much
   124.1 vel | better-auth-is-joining-vercel
   115.3 vel | streetcomplete-fixing-openstreetmap-one-tiny-quest-at-a-time
   114.0 vel | a-better-way-to-tie-your-gym-shorts-or-any-drawstring-video
    93.3 vel | europe-s-company-websites-are-mostly-served-by-us-vendors
    50.0 vel | mapping-homes-you-can-buy-from-the-us-government-for-100k


In [4]:
# ── INSPECT after Agent 1 ────────────────────────────────────────────
# Agent 1 fields: title, url, source, category, engagement, velocity
from urllib.parse import urlparse

print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
   393.2         417  technology     gamefromscratch.com          Microsoft Fire IdTech Team at Id Software
   210.6         302  technology     heise.de                     Chat Control passed first round in EU Parliam
   184.6         568  technology     whynothugo.nl                98% Isn't Much
   124.1         114  technology     better-auth.com              Better Auth is joining Vercel
   115.3         444  technology     streetcomplete.app           StreetComplete: Fixing OpenStreetMap, one tin
   114.0         392  technology     youtube.com                  A better way to tie your gym shorts. (Or any 
    93.3         329  technology     ciphercue.com                Europe's company websites are mostly served b
    50.0          34  technology     govauctions.app              Mapping homes you can buy from

---
## Agent 2: Context Researcher
For each story:
1. Fetches article content (3-tier: trafilatura → Jina → Tavily)
2. Gathers background (DDG news + Wikipedia)
3. Synthesises one tight background paragraph (qwen2.5:3b, content-anchored)

**Key design:**  rejects page chrome before accepting fetched text.
Backgrounds are honest-empty when DDG has no coverage (GitHub/arXiv/new tools — see KNOWN_ISSUES.md).

In [5]:
import ollama, subprocess, time

# Health-check: Ollama must be running for Agent 2 synthesis
try:
    ollama.list()
    print("✅ Ollama already running")
except Exception:
    try:
        subprocess.Popen(["ollama", "serve"])
        time.sleep(3)
        ollama.list()
        print("✅ Ollama started")
    except Exception:
        print("✗ Run  in a terminal manually")

✅ Ollama already running


In [6]:
from agents.agent2 import fetch_url_content, fetch_trend_background

# AGENT 2 NODE
def context_researcher_node(state: NewsroomState) -> dict:
    """Enriches each story in-place with content + background keys."""
    stories = state["stories"]
    for sid, story in stories.items():
        print(f"Agent 2 Starts For : {sid}")
        story["content"]    = fetch_url_content(story["url"])
        story["background"] = fetch_trend_background(story["title"], story["content"])
        print(f"  Agent 2 Ends for: {story['title']}")
        print("=" * 80)
    return {"stories": stories}

In [7]:
# Run Agent 2 (enriches call['stories'] in-place)
call2 = context_researcher_node(call)

Agent 2 Starts For : microsoft-fire-idtech-team-at-id-software
Jina Success ✅  In Loading Content:16251 characters
  [background] gathering snippets for: Microsoft Fire IdTech Team at Id Software
  [bg] DDG returned 1 snippets
  [bg] Wikipedia search returns: 1424 chars
  [background] 2 snippets, synthesizing (content-anchored)...
  [synth] 2 snippets, 1708 snippet chars → llama3.1:8b
  [synth] llama3.1:8b → 573 chars
Result After Background Syntezing is : id Software is a renowned game developer in the first-person genre, with their idTech technology powering numerous games and engines. The company's engine has been licensed to other developers, but it remains proprietary for id Software and its sister studios after being acquired by ZeniMax Media (later bought by Microsoft) in 2009. This era of idTech may be coming to a close as reports suggest that the team working on the engine at id Software has been significantly reduced due to massive layoffs across Xbox divisions, including fou

/Users/deepakrathore/Documents/Agentic-AI/Examples/NewsStudio/multi-agent-env/lib/python3.13/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /Users/deepakrathore/Documents/Agentic-AI/Examples/NewsStudio/multi-agent-env/lib/python3.13/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


  [bg] Wikipedia search returns: empty (skipped)
  [background] 5 snippets, synthesizing (content-anchored)...
  [synth] 5 snippets, 1146 snippet chars → llama3.1:8b
  [synth] llama3.1:8b → 492 chars
Result After Background Syntezing is : A recent box office report noted that a movie raked in $98.43 million internationally, but this statistic is misleading as it implies success for 98% of the audience, when in fact it leaves out a significant portion of viewers who may not be able to access or enjoy the film due to various technical issues. This highlights the importance of considering edge cases and ensuring that products or services can degrade gracefully, rather than relying on statistics that mask underlying problems.
  [bg] background: 492 chars
  Agent 2 Ends for: 98% Isn't Much
Agent 2 Starts For : better-auth-is-joining-vercel
Trafiltura Success ✅  In Loading Content :8755 characters
  [background] gathering snippets for: Better Auth is joining Vercel
[ddg] error for 'Better Au

In [8]:
# Run this in notebook after Agent 2 completes
first = next(iter(call2['stories'].values()))
print("Keys after Agent 2:", list(first.keys()))

Keys after Agent 2: ['title', 'url', 'source', 'category', 'engagement', 'velocity', 'content', 'background']


In [9]:
# ── INSPECT after Agent 2 ────────────────────────────────────────────
# Agent 1 fields + Agent 2 new fields: content, background
from urllib.parse import urlparse

# Agent 1 block (unchanged)
print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call2['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

# Agent 2 block (new fields only)
print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call2['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

── Agent 1 fields ──
    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
   393.2         417  technology     gamefromscratch.com          Microsoft Fire IdTech Team at Id Software
   210.6         302  technology     heise.de                     Chat Control passed first round in EU Parliam
   184.6         568  technology     whynothugo.nl                98% Isn't Much
   124.1         114  technology     better-auth.com              Better Auth is joining Vercel
   115.3         444  technology     streetcomplete.app           StreetComplete: Fixing OpenStreetMap, one tin
   114.0         392  technology     youtube.com                  A better way to tie your gym shorts. (Or any 
    93.3         329  technology     ciphercue.com                Europe's company websites are mostly served b
    50.0          34  technology     govauctions.app              Mapping h

---
## Agent 3: Fact Checker
Scores each story's credibility on a **-1 to +1 scale**.
Zero is the natural boundary — negative = discard, positive = keep.

| Signal | Base Weight | How |
|--------|-------------|-----|
| `source_score` | 20% | Domain trust map (32 domains, 0.0 to +0.95) |
| `llm_credibility_check` | 60% | gpt-oss-120b → REAL +0.9 / OPINION +0.1 / SPAM -0.7 |
| `cross_verify` | 20% | Exa → DDG → confirms or contradicts via second source |

**Dynamic reweighting** — weights shift when cross-verify fires:
- Contradiction detected → llm 60%→30%, verify 20%→50%
- Confirmation detected  → llm 60%→50%, verify 20%→30%
- Neutral (not found)    → standard weights unchanged

**Key design decisions:**
- `-1 to +1` range — negative range earned by SPAM or credible contradiction
- Threshold = 0.0 — zero is the natural semantic boundary
- SPAM → -0.7 (only genuine negative LLM signal)
- Empty response / Groq failure → 0.0 neutral (never discard on crash)
- Thin content < 500 chars → 0.0 neutral
- Soft discard — story marked `discarded=True`, never deleted (audit trail)
- Cross-verify: Exa (semantic, HN-aware) → DDG fallback → neutral
- Contradiction check uses `compound-mini` (separate quota from 120b)

In [10]:
import time                                    # ← add this line
from agents.agent3 import (
    source_score,
    llm_credibility_check,
    check_credibility,
    cross_verify,
)
print("✅ Agent 3 imported")

✅ Agent 3 imported


In [11]:
# AGENT 3 NODE
def fact_checker_node(state: NewsroomState) -> dict:
    stories = state["stories"]
    for sid, story in stories.items():
        check_credibility(story)
        flag    = "🗑️ DISCARD" if story["discarded"] else "✅ KEEP"
        regime  = story.get("_cred_regime", "?")
        vby     = story.get("verified_by", "NONE")
        contra  = "❌ contradicted" if story.get("contradicted") else ""
        print(f"{story['credibility_score']:+.2f} {flag} "
              f"[{regime}] verified={vby} {contra}| {story['title']}")
        time.sleep(1)
    return {"stories": stories}

In [12]:
# Run Agent 3 on Agent 2's output (no re-run of A1 or A2)
print("── CREDIBILITY RESULTS ─────────────────────────────")
call3 = fact_checker_node(call2)
print("────────────────────────────────────────────────────")

── CREDIBILITY RESULTS ─────────────────────────────
  [llm_cred_check] Groq failed: Error code: 403 - {'error': {'message': 'Access denied. Please check your network settings.'}} → 0.0 neutral
  [verify] neutral — no credible source found or both tiers failed
  [cred] regime=neutral | src=0.00×0.2 + llm=0.00×0.6 + verify=0.00×0.2 = 0.00
+0.00 ✅ KEEP [neutral] verified=NONE | Microsoft Fire IdTech Team at Id Software
  [llm_cred_check] Groq failed: Error code: 403 - {'error': {'message': 'Access denied. Please check your network settings.'}} → 0.0 neutral
  [verify] neutral — no credible source found or both tiers failed
  [cred] regime=neutral | src=0.00×0.2 + llm=0.00×0.6 + verify=0.00×0.2 = 0.00
+0.00 ✅ KEEP [neutral] verified=NONE | Chat Control passed first round in EU Parliament
  [llm_cred_check] Groq failed: Error code: 403 - {'error': {'message': 'Access denied. Please check your network settings.'}} → 0.0 neutral
  [verify] neutral — no credible source found or both tiers fai

In [13]:
# Run this after Agent 3 completes
first = next(iter(call3['stories'].values()))
print("Keys after Agent 3:", list(first.keys()))

Keys after Agent 3: ['title', 'url', 'source', 'category', 'engagement', 'velocity', 'content', 'background', 'credibility_score', 'verified_by', 'contradicted', 'discarded', '_cred_regime', '_weights_used']


In [14]:
for sid, story in call3["stories"].items():
    print(sid)
    print(story)

microsoft-fire-idtech-team-at-id-software
{'title': 'Microsoft Fire IdTech Team at Id Software', 'url': 'https://gamefromscratch.com/microsoft-fire-idtech-team-at-id-software/', 'source': 'hackernews', 'category': 'technology', 'engagement': 417, 'velocity': 393.2, 'content': '[Skip to content](https://gamefromscratch.com/microsoft-fire-idtech-team-at-id-software/#content)\n\n[![Image 5: GameFromScratch.com](https://gamefromscratch.com/wp-content/uploads/2020/07/cropped-FloppyLogoBlue-1-e1596321567124-90x89.png)](https://gamefromscratch.com/)\n\n[GameFromScratch.com](https://gamefromscratch.com/)\nGame Development News, Tutorials and More\n\nMenu Menu\n\n*   [News](https://gamefromscratch.com/news/)\n*   [Reviews](https://gamefromscratch.com/reviews/)\n*   [Tutorials](https://gamefromscratch.com/tutorials/)\n*   [Resources](https://gamefromscratch.com/resources/)\n*   Search for: [Search](https://gamefromscratch.com/microsoft-fire-idtech-team-at-id-software/#)  \n\n*   [YouTube](https:

In [15]:
# ── INSPECT after Agent 3 — cumulative (A1 + A2 + A3 fields) ────────
# Keys added: credibility_score, verified_by, contradicted,
#             discarded, _cred_regime, _weights_used
from urllib.parse import urlparse

print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call3['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call3['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

print()
print('── Agent 3 new fields ──')
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'REGIME':<14} {'WEIGHTS':<12} "
      f"{'VERIFIED_BY':<22} {'SOURCE_DOMAIN':<25} TITLE")
print('-' * 120)
for sid, story in call3['stories'].items():
    flag    = '🗑️' if story.get('discarded') else '✅'
    score   = story.get('credibility_score', 0)
    vel     = story.get('velocity', 0)
    regime  = story.get('_cred_regime', '?')
    weights = story.get('_weights_used', '?')
    vby     = story.get('verified_by', 'NONE')
    contra  = '❌ ' if story.get('contradicted') else ''
    domain  = urlparse(story.get('url','')).netloc.replace('www.','')
    title   = story.get('title','')[:40]
    print(f"{flag}  {score:+.2f}  {vel:>6.1f}  {regime:<14} {weights:<12} "
          f"{contra}{vby:<22} {domain:<25} {title}")


── Agent 1 fields ──
    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
   393.2         417  technology     gamefromscratch.com          Microsoft Fire IdTech Team at Id Software
   210.6         302  technology     heise.de                     Chat Control passed first round in EU Parliam
   184.6         568  technology     whynothugo.nl                98% Isn't Much
   124.1         114  technology     better-auth.com              Better Auth is joining Vercel
   115.3         444  technology     streetcomplete.app           StreetComplete: Fixing OpenStreetMap, one tin
   114.0         392  technology     youtube.com                  A better way to tie your gym shorts. (Or any 
    93.3         329  technology     ciphercue.com                Europe's company websites are mostly served b
    50.0          34  technology     govauctions.app              Mapping h

In [16]:
from urllib.parse import urlparse
# ── INSPECT: full story details after all 3 agents ──────────────────
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'CONTENT':>8} {'BG':>5} {'REGIME':<14} {'VERIFIED_BY':<19} {'ORIGINAL SOURCE':<25} {'TITLE':<50}")
print("-" * 145)



for sid, story in call3["stories"].items():
    flag    = "🗑️" if story.get("discarded") else "✅"
    score   = story.get("credibility_score", 0)
    vel     = story.get("velocity", 0)
    clen    = len(story.get("content", ""))
    bglen   = len(story.get("background", ""))
    regime  = story.get("_cred_regime", "?")
    vby     = story.get("verified_by", "NONE")
    contra  = "❌" if story.get("contradicted") else ""
    #weights = story.get("_weights_used", "?")
    title   = story.get("title", "")
    source = story.get("url", "")
    source_domain = urlparse(
            source
        ).netloc.replace("www.", "")


    print(    f"{flag}  {score:+.2f}  {vel:>6.1f}  {clen:>6}c  "
            f"{bglen:>4}c  {regime:<14} {contra}{vby:<20}"
            f"{source_domain:<25} {title:<50}")

FLAG SCORE      VEL  CONTENT    BG REGIME         VERIFIED_BY         ORIGINAL SOURCE           TITLE                                             
-------------------------------------------------------------------------------------------------------------------------------------------------
✅  +0.00   393.2    6000c   573c  neutral        NONE                gamefromscratch.com       Microsoft Fire IdTech Team at Id Software         
✅  +0.00   210.6    4320c   618c  neutral        NONE                heise.de                  Chat Control passed first round in EU Parliament  
✅  +0.00   184.6    2096c   492c  neutral        NONE                whynothugo.nl             98% Isn't Much                                    
✅  +0.24   124.1    6000c   798c  confirmation   github.com          better-auth.com           Better Auth is joining Vercel                     
✅  +0.24   115.3     343c   572c  confirmation   github.com          streetcomplete.app        StreetComplete: Fixing OpenS

### Save story : till agent 3 (quality data progress saved to disk) so even if some changes comes in agent 1 to agent 3 , agent 4+ code developement will not be affect

In [17]:

# Save to disk — skip re-processing next time
#from agent_tools.story_cache import save_stories
#save_stories(call3["stories"])

# TO track log of particular function after a hit of particular no. of times pop-up will come


---
## Agent 4: Editorial

Selects the top stories to cover today from Agent 3's credibility-scored pool.

**Responsibilities:**
1. **Filter** — remove Agent 3 discards (credibility_score < 0.0)
2. **Score** — editorial_score = cred×0.50 + vel_norm×0.30 + bg_norm×0.20
3. **Deduplicate** — phi3.5 clusters titles by topic (one call, no bleed)
4. **Select** — top 3 by editorial_score with topic diversity enforced
5. **Route** — ≥1 story → Agent 5 · 0 stories → end + macOS notification

**First conditional edge in the pipeline** — all previous agents were linear.

In [18]:
from agents.agent4 import (
    editorial_node,
    route_after_editorial,
)

# Run Agent 4 on Agent 3's output
print("=" * 70)
call4 = editorial_node(call3)
print()

# Show routing decision
route = route_after_editorial(call4)
print(f"\n  Pipeline continues to: {route}")

AGENT 4: Editorial
  [filter] 8 stories in → 8 eligible (0 discarded by Agent 3)

  [editorial] max velocity this batch: 393.2
  [editorial] Microsoft Fire IdTech Team at Id Software     cred=+0.00 vel=1.00 bg=0.72 → 0.443
  [editorial] Chat Control passed first round in EU Parliam cred=+0.00 vel=0.54 bg=0.77 → 0.315
  [editorial] 98% Isn't Much                                cred=+0.00 vel=0.47 bg=0.61 → 0.264
  [editorial] Better Auth is joining Vercel                 cred=+0.24 vel=0.32 bg=1.00 → 0.414
  [editorial] StreetComplete: Fixing OpenStreetMap, one tin cred=+0.24 vel=0.29 bg=0.71 → 0.351
  [editorial] A better way to tie your gym shorts. (Or any  cred=+0.00 vel=0.29 bg=0.59 → 0.205
  [editorial] Europe's company websites are mostly served b cred=+0.00 vel=0.24 bg=0.90 → 0.251
  [editorial] Mapping homes you can buy from the US governm cred=+0.00 vel=0.13 bg=0.52 → 0.142

  [deduplicate] phi3.5 raw: [
  [1, 3],
  [2, 5],
  [4],
  [6],
  [7],
  [8]
]
  [deduplicate] 6 topic c

In [21]:
# ── INSPECT after Agent 4 — cumulative (A1 + A2 + A3 + A4 fields) ───
# Keys added: editorial_score, selected, selection_rank,
#             selection_reason, _vel_norm, _bg_norm,
#             _topic_cluster, _is_duplicate
from urllib.parse import urlparse

print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call4['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call4['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

print()
print('── Agent 3 new fields ──')
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'REGIME':<14} {'WEIGHTS':<12} "
      f"{'VERIFIED_BY':<22} {'SOURCE_DOMAIN':<25} TITLE")
print('-' * 120)
for sid, story in call4['stories'].items():
    flag    = '🗑️' if story.get('discarded') else '✅'
    score   = story.get('credibility_score', 0)
    vel     = story.get('velocity', 0)
    regime  = story.get('_cred_regime', '?')
    weights = story.get('_weights_used', '?')
    vby     = story.get('verified_by', 'NONE')
    contra  = '❌ ' if story.get('contradicted') else ''
    domain  = urlparse(story.get('url','')).netloc.replace('www.','')
    title   = story.get('title','')[:40]
    print(f"{flag}  {score:+.2f}  {vel:>6.1f}  {regime:<14} {weights:<12} "
          f"{contra}{vby:<22} {domain:<25} {title}")

print()
print('── Agent 4 new fields ──')
print(f"{'SEL':<5} {'RANK':<5} {'ED_SCORE':<10} {'VEL_N':<7} {'BG_N':<7} "
      f"{'CLUSTER':<9} {'IS_DUPLICATE':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 110)
for sid, story in call4['stories'].items():
    selected  = '✅' if story.get('selected') else '—'
    rank      = story.get('selection_rank', '-')
    ed_score  = story.get('editorial_score', 0)
    vel_n     = story.get('_vel_norm', 0)
    bg_n      = story.get('_bg_norm', 0)
    cluster   = story.get('_topic_cluster', '-')
    is_dup    = '❌' if story.get('_is_duplicate') else '✅'
    domain    = urlparse(story.get('url','')).netloc.replace('www.','')
    title     = story.get('title','')[:40]
    print(f"{selected:<5} #{rank!s:<4} {ed_score:<10.3f} {vel_n:<7.3f} {bg_n:<7.3f} "
          f"{cluster!s:<9} {is_dup:<10} {domain:<28} {title}")

── Agent 1 fields ──
    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
   393.2         417  technology     gamefromscratch.com          Microsoft Fire IdTech Team at Id Software
   210.6         302  technology     heise.de                     Chat Control passed first round in EU Parliam
   184.6         568  technology     whynothugo.nl                98% Isn't Much
   124.1         114  technology     better-auth.com              Better Auth is joining Vercel
   115.3         444  technology     streetcomplete.app           StreetComplete: Fixing OpenStreetMap, one tin
   114.0         392  technology     youtube.com                  A better way to tie your gym shorts. (Or any 
    93.3         329  technology     ciphercue.com                Europe's company websites are mostly served b
    50.0          34  technology     govauctions.app              Mapping h